# 📈 StokSignal — Прогноз цен акций MOEX с ML
**ИТ-стартап | Финансовый университет | Технологии ИИ**

Пайплайн: загрузка ISS MOEX → очистка → feature engineering → GBR/RF/LR → оценка → торговый сигнал → SQLite

---


## ⚙️ Установка зависимостей

In [ ]:
# Устанавливаем aiomoex (клиент ISS MOEX) — всё остальное уже есть в Colab
!pip install aiomoex --quiet

import importlib, sys
for pkg in ['aiomoex', 'pandas', 'numpy', 'sklearn', 'matplotlib', 'seaborn', 'sqlite3']:
    try:
        importlib.import_module(pkg)
        print(f'  ✓ {pkg}')
    except ImportError:
        print(f'  ✗ {pkg} — НЕТ')
print('\nВсе зависимости проверены.')

---
## 📥 Модуль 1 — Загрузка данных с ISS MOEX

Асинхронная загрузка исторических котировок через `aiomoex`.
Тикеры: **SBER** (Сбербанк), **GAZP** (Газпром), **LKOH** (Лукойл).
Период: **2019-01-01 — 2024-12-31** (~1300 торговых дней на тикер).


In [ ]:
import asyncio
import aiohttp
import aiomoex
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from datetime import date as _date

# ── Параметры ──────────────────────────────────────────────────────────
TICKERS    = ['SBER', 'GAZP', 'LKOH']
DATE_END   = _date.today().isoformat()   # всегда актуальная дата
COLUMNS    = ['OPEN', 'HIGH', 'LOW', 'CLOSE', 'VOLUME']

# ГАЗП сделал сплит акций 1:40 в июле 2022:
# до сплита цена ~300–400 руб., после ~7–15 руб.
# Берём данные только ПОСЛЕ сплита, иначе модель обучится
# на старых ценах и будет предсказывать 3000+ при реальных 100 руб.
TICKER_START = {
    'SBER': '2019-01-01',
    'GAZP': '2022-08-01',   # после сплита 1:40
    'LKOH': '2019-01-01',
}

async def load_ticker(session: aiohttp.ClientSession, ticker: str) -> pd.DataFrame:
    """Загружает историю тикера с MOEX ISS начиная с нужной даты."""
    data = await aiomoex.get_board_history(
        session, ticker,
        start=TICKER_START[ticker],
        end=DATE_END,
        columns=['TRADEDATE'] + COLUMNS
    )
    if not data:
        raise ValueError(f'Нет данных для тикера {ticker}')
    df = pd.DataFrame(data)
    df['TRADEDATE'] = pd.to_datetime(df['TRADEDATE'])
    df = df.set_index('TRADEDATE').sort_index()
    df = df[COLUMNS].astype(float)
    return df

async def load_all_tickers() -> dict:
    """Загружает все тикеры асинхронно за один сеанс."""
    async with aiohttp.ClientSession() as session:
        results = {}
        for ticker in TICKERS:
            results[ticker] = await load_ticker(session, ticker)
    return results

# ── Запуск загрузки ────────────────────────────────────────────────────
import time
print('Загружаем данные с MOEX ISS...')
t0 = time.time()

try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

raw_data = asyncio.run(load_all_tickers())
elapsed = time.time() - t0

print(f'\n✅ Загружено за {elapsed:.1f} сек.')
print('-' * 65)
for ticker, df in raw_data.items():
    print(f'  {ticker}: {len(df)} торговых дней  '
          f'({df.index[0].date()} → {df.index[-1].date()})  '
          f'цена: {df["CLOSE"].min():.1f}–{df["CLOSE"].max():.1f} руб.')

# ── Контроль адекватности цен ──────────────────────────────────────────
print('\nПроверка диапазонов цен (последние 30 дней):')
for ticker, df in raw_data.items():
    recent = df['CLOSE'].tail(30)
    print(f'  {ticker}: среднее {recent.mean():.2f} руб.  '
          f'мин {recent.min():.2f}  макс {recent.max():.2f}')


---
## 🧹 Модуль 2 — Очистка данных

Применяем три контроля качества:
1. **Пропуски** — `ffill` (перенос последнего известного значения)
2. **Выбросы** — метод IQR: замена медианой скользящего окна 20 дней
3. **Нормализация** — `log1p(VOLUME)` → `VOLUME_LOG`

Затем разбиваем на **train (80%) / test (20%)** строго по времени (без shuffle).


In [ ]:
def remove_outliers_iqr(series: pd.Series, window: int = 20) -> pd.Series:
    """Заменяет выбросы (IQR-метод) медианой скользящего окна."""
    s = series.copy()
    q1 = s.rolling(window, min_periods=5).quantile(0.25)
    q3 = s.rolling(window, min_periods=5).quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    med = s.rolling(window, min_periods=5).median()
    mask = (s < lower) | (s > upper)
    s[mask] = med[mask]
    return s

def clean_ticker(df: pd.DataFrame) -> pd.DataFrame:
    """Полная очистка одного тикера."""
    df = df.copy()
    # 1. Пропуски
    before = df.isnull().sum().sum()
    df = df.ffill().dropna()
    after_miss = df.isnull().sum().sum()
    # 2. Выбросы по CLOSE и VOLUME
    df['CLOSE']  = remove_outliers_iqr(df['CLOSE'])
    df['VOLUME'] = remove_outliers_iqr(df['VOLUME'])
    # 3. Логарифм объёма
    df['VOLUME_LOG'] = np.log1p(df['VOLUME'])
    return df

# ── Применяем ко всем тикерам ─────────────────────────────────────────
clean_data = {t: clean_ticker(df) for t, df in raw_data.items()}

# ── Разбивка train / test (80 / 20) без перемешивания ─────────────────
TRAIN_RATIO = 0.80

splits = {}
for ticker, df in clean_data.items():
    n = len(df)
    split_idx = int(n * TRAIN_RATIO)
    splits[ticker] = {
        'df':    df,
        'train': df.iloc[:split_idx],
        'test':  df.iloc[split_idx:],
    }

print('Результаты очистки и разбивки:')
print('-' * 65)
for ticker, s in splits.items():
    tr, te = s['train'], s['test']
    print(f'  {ticker}: всего {len(s["df"])} дн. | '
          f'train {len(tr)} ({tr.index[0].date()}–{tr.index[-1].date()}) | '
          f'test {len(te)} ({te.index[0].date()}–{te.index[-1].date()})')
print('\nПример очищенных данных SBER:')
print(clean_data['SBER'][['OPEN','HIGH','LOW','CLOSE','VOLUME','VOLUME_LOG']].tail(5).round(2).to_string())

---
## 🔧 Модуль 3 — Feature Engineering

Строим **11 признаков** для каждого тикера:

| Признак | Описание |
|---------|----------|
| `Lag_1, 5, 10` | Цена закрытия 1/5/10 дней назад |
| `MA_5, MA_20` | Скользящее среднее 5 и 20 дней |
| `STD_10` | Волатильность за 10 дней |
| `RSI_14` | Индекс относительной силы |
| `Return_1d` | Дневная доходность |
| `DayOfWeek, Month` | Календарные признаки |
| `VOLUME_LOG` | Логарифм объёма |


In [ ]:
def compute_rsi(series: pd.Series, period: int = 14) -> pd.Series:
    """RSI (Relative Strength Index)."""
    delta = series.diff()
    gain  = delta.clip(lower=0).rolling(period).mean()
    loss  = (-delta.clip(upper=0)).rolling(period).mean()
    rs    = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

FEATURES = ['Lag_1','Lag_5','Lag_10',
            'MA_5','MA_20','STD_10',
            'RSI_14','Return_1d',
            'DayOfWeek','Month','VOLUME_LOG']
TARGET = 'CLOSE'

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """Добавляет все признаки, удаляет NaN."""
    d = df.copy()
    d['Lag_1']     = d['CLOSE'].shift(1)
    d['Lag_5']     = d['CLOSE'].shift(5)
    d['Lag_10']    = d['CLOSE'].shift(10)
    d['MA_5']      = d['CLOSE'].rolling(5).mean()
    d['MA_20']     = d['CLOSE'].rolling(20).mean()
    d['STD_10']    = d['CLOSE'].rolling(10).std()
    d['RSI_14']    = compute_rsi(d['CLOSE'], 14)
    d['Return_1d'] = d['CLOSE'].pct_change()
    d['DayOfWeek'] = d.index.dayofweek
    d['Month']     = d.index.month
    return d.dropna(subset=FEATURES + [TARGET])

feature_data = {t: add_features(s['df']) for t, s in splits.items()}

print('Признаки построены:')
print('-' * 50)
for ticker, df in feature_data.items():
    print(f'  {ticker}: {len(df)} строк, {len(FEATURES)} признаков')
print()
print('Пример признаков для SBER (последние 3 строки):')
print(feature_data['SBER'][FEATURES + [TARGET]].tail(3).round(2).to_string())

---
## 🤖 Модуль 4 — Обучение и сравнение моделей

Сравниваем **4 модели** на каждом тикере:
- **Linear Regression** — baseline
- **Random Forest Regressor** — ансамблевый метод
- **Gradient Boosting Regressor (GBR)** — финальная модель для сигналов ★
- **LSTM** — рекуррентная нейронная сеть для временных рядов

Разбивка: строго train/test по времени (первые 80% = train).


In [ ]:
from sklearn.linear_model  import LinearRegression
from sklearn.ensemble      import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics       import mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# TensorFlow/Keras — уже установлен в Colab
import tensorflow as tf
from tensorflow.keras.models    import Sequential
from tensorflow.keras.layers    import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
tf.get_logger().setLevel('ERROR')

def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

# ── Строим LSTM-последовательности (окно 20 дней) ─────────────────────
LSTM_WINDOW = 20

def build_lstm_sequences(X: np.ndarray, y: np.ndarray, window: int):
    """Разбивает данные на overlapping окна для LSTM."""
    Xs, ys = [], []
    for i in range(len(X) - window):
        Xs.append(X[i:i + window])
        ys.append(y[i + window])
    return np.array(Xs), np.array(ys)

def build_lstm_model(n_features: int, window: int) -> Sequential:
    model = Sequential([
        LSTM(64, input_shape=(window, n_features), return_sequences=True),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

# ── Обучение всех моделей ─────────────────────────────────────────────
SPLIT  = 0.80
results  = {}
trained  = {}
preds    = {}
lstm_scalers = {}   # MinMaxScaler для каждого тикера (LSTM требует нормализацию)

for ticker, df in feature_data.items():
    X = df[FEATURES].values
    y = df[TARGET].values
    n = len(X)
    split_idx = int(n * SPLIT)

    X_train_raw, X_test_raw = X[:split_idx], X[split_idx:]
    y_train,     y_test     = y[:split_idx], y[split_idx:]

    results[ticker] = {}
    trained[ticker] = {}
    preds[ticker]   = {'y_test': df[TARGET].iloc[split_idx:]}

    # ── 1. Linear Regression ──────────────────────────────────────
    lr = LinearRegression()
    lr.fit(X_train_raw, y_train)
    yp = lr.predict(X_test_raw)
    results[ticker]['Linear Regression'] = {
        'MAE':  round(mean_absolute_error(y_test, yp), 2),
        'MAPE': round(mape(y_test, yp), 2),
        'R2':   round(r2_score(y_test, yp), 4),
    }
    trained[ticker]['Linear Regression'] = lr
    preds[ticker]['Linear Regression']   = yp

    # ── 2. Random Forest ──────────────────────────────────────────
    rf = RandomForestRegressor(n_estimators=200, max_depth=8,
                                random_state=42, n_jobs=-1)
    rf.fit(X_train_raw, y_train)
    yp = rf.predict(X_test_raw)
    results[ticker]['Random Forest'] = {
        'MAE':  round(mean_absolute_error(y_test, yp), 2),
        'MAPE': round(mape(y_test, yp), 2),
        'R2':   round(r2_score(y_test, yp), 4),
    }
    trained[ticker]['Random Forest'] = rf
    preds[ticker]['Random Forest']   = yp

    # ── 3. GBR ★ ─────────────────────────────────────────────────
    gbr = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                     max_depth=4, subsample=0.8, random_state=42)
    gbr.fit(X_train_raw, y_train)
    yp = gbr.predict(X_test_raw)
    results[ticker]['GBR ★'] = {
        'MAE':  round(mean_absolute_error(y_test, yp), 2),
        'MAPE': round(mape(y_test, yp), 2),
        'R2':   round(r2_score(y_test, yp), 4),
    }
    trained[ticker]['GBR ★'] = gbr
    preds[ticker]['GBR ★']   = yp

    # ── 4. LSTM ───────────────────────────────────────────────────
    # Нормализуем признаки для LSTM
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()
    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).ravel()
    lstm_scalers[ticker] = {'X': scaler_X, 'y': scaler_y}

    X_seq, y_seq = build_lstm_sequences(X_scaled, y_scaled, LSTM_WINDOW)

    # Пересчитываем split для последовательностей (они короче на LSTM_WINDOW)
    split_seq = int(len(X_seq) * SPLIT)
    X_tr_s, X_te_s = X_seq[:split_seq], X_seq[split_seq:]
    y_tr_s, y_te_s = y_seq[:split_seq], y_seq[split_seq:]

    lstm = build_lstm_model(n_features=len(FEATURES), window=LSTM_WINDOW)
    es   = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    lstm.fit(X_tr_s, y_tr_s,
             epochs=50, batch_size=32,
             validation_split=0.1,
             callbacks=[es], verbose=0)

    yp_scaled = lstm.predict(X_te_s, verbose=0).ravel()
    yp_lstm   = scaler_y.inverse_transform(yp_scaled.reshape(-1, 1)).ravel()
    # y_test для LSTM чуть короче из-за окна
    y_test_lstm = y[split_idx + LSTM_WINDOW:]

    results[ticker]['LSTM'] = {
        'MAE':  round(mean_absolute_error(y_test_lstm, yp_lstm), 2),
        'MAPE': round(mape(y_test_lstm, yp_lstm), 2),
        'R2':   round(r2_score(y_test_lstm, yp_lstm), 4),
    }
    trained[ticker]['LSTM'] = {'model': lstm, 'scaler_X': scaler_X, 'scaler_y': scaler_y}
    preds[ticker]['LSTM']   = yp_lstm

    print(f"  {ticker} готов")

# ── Сводная таблица метрик ─────────────────────────────────────────────
print()
print('=' * 68)
print(f'{"Тикер":<8} {"Модель":<22} {"MAE":>8} {"MAPE":>8} {"R²":>8}')
print('=' * 68)
for ticker in TICKERS:
    for name, m in results[ticker].items():
        marker = ' ←' if '★' in name else ''
        print(f'{ticker:<8} {name:<22} {m["MAE"]:>8.2f} '
              f'{m["MAPE"]:>7.2f}% {m["R2"]:>8.4f}{marker}')
    print('-' * 68)
print('\n★  GBR — финальная модель для сигналов (лучший баланс точности и скорости)')
print('   LSTM — показывает альтернативную оценку на нейросетевом подходе')


---
## 📊 Модуль 5 — Оценка качества и визуализация

Строим три блока графиков:
1. **Прогноз vs факт** — GBR и LSTM по каждому тикеру на тестовой выборке
2. **Важность признаков** — feature_importances_ для GBR (SBER)
3. **Сводная таблица** метрик всех 4 моделей


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.0)

# ── 1. Прогноз vs факт: GBR и LSTM по 3 тикерам ───────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharey=False)
fig.suptitle('StokSignal — прогноз GBR и LSTM vs факт (тестовая выборка)',
             fontsize=13, fontweight='bold')

for col, ticker in enumerate(TICKERS):
    y_test   = preds[ticker]['y_test']
    y_gbr    = preds[ticker]['GBR ★']
    y_lstm   = preds[ticker]['LSTM']
    dates_gbr  = y_test.index
    # LSTM на LSTM_WINDOW точек короче
    dates_lstm = y_test.index[LSTM_WINDOW:]

    for row, (y_pred, label, color, ax_row) in enumerate([
        (y_gbr,  'GBR ★',  '#E07B39', 0),
        (y_lstm, 'LSTM',   '#7B5EA7', 1),
    ]):
        ax = axes[ax_row][col]
        dates = dates_gbr if label == 'GBR ★' else dates_lstm
        y_true = y_test.values if label == 'GBR ★' else y_test.values[LSTM_WINDOW:]

        ax.plot(dates, y_true,  label='Факт',    color='#2C3E50', lw=1.4, alpha=0.9)
        ax.plot(dates, y_pred,  label=label,     color=color,     lw=1.2,
                linestyle='--', alpha=0.85)
        ax.fill_between(dates, y_true, y_pred, alpha=0.07, color=color)

        m = results[ticker][label]
        ax.set_title(f'{ticker} / {label}  MAPE={m["MAPE"]}%  R²={m["R2"]}',
                     fontsize=9)
        ax.set_xlabel('Дата', fontsize=8)
        ax.legend(fontsize=7)
        ax.xaxis.set_major_locator(mticker.MaxNLocator(3))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.show()

# ── 2. Важность признаков (SBER, GBR) ─────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
importances = (
    __import__('pandas').Series(
        trained['SBER']['GBR ★'].feature_importances_,
        index=FEATURES
    ).sort_values()
)
bars = ax.barh(importances.index, importances.values * 100,
               color='#E07B39', edgecolor='none', alpha=0.85)
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=8)
ax.set_xlabel('Важность признака, %')
ax.set_title('Важность признаков GBR — тикер SBER', fontweight='bold')
ax.set_xlim(0, importances.max() * 100 * 1.18)
plt.tight_layout()
plt.show()

# ── 3. Сводная таблица метрик ──────────────────────────────────────────
import pandas as pd
rows = []
for ticker in TICKERS:
    for name, m in results[ticker].items():
        rows.append({'Тикер': ticker, 'Модель': name,
                     'MAE': m['MAE'], 'MAPE': str(m['MAPE'])+'%', 'R²': m['R2']})
summary = pd.DataFrame(rows)
print('\nСводная таблица метрик:')
print(summary.to_string(index=False))


---
## 🚦 Модуль 6 — Торговый сигнал

Логика: если прогнозируемое изменение цены превышает порог **±3%** → генерируем сигнал **BUY** или **SELL**, иначе **HOLD**.

Выводим таблицу последних 20 сигналов для каждого тикера.


In [ ]:
SIGNAL_THRESHOLD = 0.03    # 3%  — порог сигнала
MAX_DELTA_PCT    = 50.0    # >50% — явный баг данных, игнорируем

def generate_signals(ticker: str) -> pd.DataFrame:
    """Генерирует торговые сигналы на тестовой выборке."""
    df = feature_data[ticker].copy()
    n  = len(df)
    split_idx = int(n * SPLIT)
    df_test = df.iloc[split_idx:].copy()

    X_test = df_test[FEATURES]
    y_pred = trained[ticker]['GBR ★'].predict(X_test)

    df_s = df_test[[TARGET]].copy()
    df_s.rename(columns={TARGET: 'Цена_факт'}, inplace=True)
    df_s['Цена_прогноз'] = y_pred.round(2)
    df_s['Изменение_%']  = ((df_s['Цена_прогноз'] - df_s['Цена_факт'])
                             / df_s['Цена_факт'] * 100).round(2)

    # Фильтр аномалий: если дельта > MAX_DELTA_PCT — данные некорректны
    df_s['Сигнал'] = df_s['Изменение_%'].apply(
        lambda d: '⚠️ ДАННЫЕ'  if abs(d) > MAX_DELTA_PCT
             else '🟢 BUY'     if d >  SIGNAL_THRESHOLD * 100
             else '🔴 SELL'    if d < -SIGNAL_THRESHOLD * 100
             else '⚪ HOLD'
    )
    df_s['Тикер'] = ticker
    return df_s

# ── Генерируем сигналы для всех тикеров ──────────────────────────────
all_signals = {}
print('Торговые сигналы (последние 10 дней на тикер):')
print('=' * 70)

for ticker in TICKERS:
    sigs = generate_signals(ticker)
    all_signals[ticker] = sigs

    # Статистика
    buy_n    = (sigs['Сигнал'] == '🟢 BUY').sum()
    sell_n   = (sigs['Сигнал'] == '🔴 SELL').sum()
    hold_n   = (sigs['Сигнал'] == '⚪ HOLD').sum()
    anomaly_n = sigs['Сигнал'].str.contains('ДАННЫЕ').sum()

    print(f'\n  {ticker}  |  BUY: {buy_n}  SELL: {sell_n}  '
          f'HOLD: {hold_n}  Аномалий: {anomaly_n}')

    if anomaly_n > 0:
        print(f'  ⚠️  Обнаружено {anomaly_n} аномальных строк — '
              f'проверьте данные (сплит/корп. действия)')

    print('  ' + '-' * 60)
    display_cols = ['Тикер','Цена_факт','Цена_прогноз','Изменение_%','Сигнал']
    # Показываем только нормальные строки в последних 10 днях
    clean_sigs = sigs[~sigs['Сигнал'].str.contains('ДАННЫЕ')]
    print(clean_sigs[display_cols].tail(10).to_string())


---
## 💾 Модуль 7 — Сохранение модели (.pkl)

Сохраняем обученную GBR-модель, scaler признаков и список фич в файл `stocsignal_model.pkl`.  
Этот файл потом подключается к Telegram-боту — бот загружает его один раз и использует для ежедневных прогнозов.


In [ ]:
import pickle, os

# Сохраняем GBR и LSTM для всех тикеров в один bundle
model_bundle = {
    "models":        {ticker: trained[ticker]["GBR ★"]  for ticker in TICKERS},
    "lstm_models":   {ticker: trained[ticker]["LSTM"]    for ticker in TICKERS},
    "features":      FEATURES,
    "target":        TARGET,
    "threshold":     0.03,
    "lstm_window":   LSTM_WINDOW,
    "tickers":       TICKERS,
    "saved_at":      str(__import__("pandas").Timestamp.today().date()),
    "last_data":     {t: feature_data[t].tail(30) for t in TICKERS},
}

MODEL_PATH = "stocsignal_model.pkl"
with open(MODEL_PATH, "wb") as f:
    pickle.dump(model_bundle, f)

size_kb = os.path.getsize(MODEL_PATH) / 1024
print(f"✅ Модель сохранена: {MODEL_PATH}  ({size_kb:.0f} KB)")
print(f"   Тикеры: {TICKERS}")
print(f"   GBR + LSTM для каждого тикера")
print(f"   Признаков: {len(FEATURES)}")
print(f"   Дата обучения: {model_bundle['saved_at']}")
print()

with open(MODEL_PATH, "rb") as f:
    check = pickle.load(f)
print(f"✅ Проверка: GBR моделей: {len(check['models'])}, LSTM моделей: {len(check['lstm_models'])}")
print()
print("📥 Скачайте stocsignal_model.pkl через панель Files в Colab (левая панель → Files)")


---
## 🤖 Модуль 8 — Telegram-бот (запускается локально, НЕ в Colab)

> ⚠️ **Эту ячейку не запускайте в Colab** — скопируйте код в файл `stocsignal_bot.py` на своём компьютере или VPS.

**Установка зависимостей на локальной машине:**
```
pip install python-telegram-bot apscheduler aiomoex aiohttp scikit-learn pandas numpy
```

**Что делает бот:**
- Загружает `stocsignal_model.pkl` (скачайте из Colab через Files)
- Каждый день в 09:00 МСК сам забирает котировки с MOEX
- Считает прогноз через обученную модель
- Отправляет сигнал BUY / SELL / HOLD в Telegram

**Подготовка:**
1. Создайте бота через @BotFather → скопируйте `BOT_TOKEN`
2. Узнайте свой `CHAT_ID` через @userinfobot
3. Вставьте оба значения в код ниже вместо `"ВАШ_ТОКЕН"` и `"ВАШ_CHAT_ID"`


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ЭТОТ КОД — ДЛЯ ЛОКАЛЬНОГО ЗАПУСКА. НЕ ЗАПУСКАЙТЕ В COLAB.     ║
# ║  Скопируйте в файл stocsignal_bot.py и запустите на компьютере. ║
# ╚══════════════════════════════════════════════════════════════════╝

import sys, os

# Защита от случайного запуска в Colab
if 'google.colab' in sys.modules or os.path.exists('/content'):
    print("=" * 60)
    print("⛔  Эта ячейка предназначена для локального запуска.")
    print()
    print("Что сделать:")
    print("  1. Скачайте stocsignal_model.pkl из панели Files (слева)")
    print("  2. Скопируйте код ниже в файл stocsignal_bot.py")
    print("  3. Запустите на своём компьютере или VPS:")
    print("     pip install python-telegram-bot apscheduler aiomoex")
    print("     python stocsignal_bot.py")
    print("=" * 60)
else:
    # ── Настройки — замените на свои ──────────────────────────────
    BOT_TOKEN  = "ВАШ_ТОКЕН_ОТ_BOTFATHER"   # @BotFather → /newbot
    CHAT_ID    = "ВАШ_CHAT_ID"              # @userinfobot
    MODEL_PATH = "stocsignal_model.pkl"

    import pickle, asyncio, logging
    from datetime import date
    import pandas as pd
    import numpy as np
    import aiohttp, aiomoex
    from telegram import Bot
    from apscheduler.schedulers.asyncio import AsyncIOScheduler

    logging.basicConfig(level=logging.INFO,
                        format="%(asctime)s | %(levelname)s | %(message)s")

    # Загрузка модели
    with open(MODEL_PATH, "rb") as f:
        BUNDLE = pickle.load(f)

    MODELS    = BUNDLE["models"]
    FEATURES  = BUNDLE["features"]
    TICKERS   = BUNDLE["tickers"]
    THRESHOLD = BUNDLE["threshold"]

    TICKER_START = {"SBER": "2023-01-01", "GAZP": "2023-01-01", "LKOH": "2023-01-01"}

    def compute_rsi(series, period=14):
        delta = series.diff()
        gain  = delta.clip(lower=0).rolling(period).mean()
        loss  = (-delta.clip(upper=0)).rolling(period).mean()
        rs    = gain / loss.replace(0, np.nan)
        return 100 - (100 / (1 + rs))

    async def fetch_ticker(session, ticker):
        data = await aiomoex.get_board_history(
            session, ticker,
            start=TICKER_START[ticker],
            end=date.today().isoformat(),
            columns=["TRADEDATE","OPEN","HIGH","LOW","CLOSE","VOLUME"]
        )
        df = pd.DataFrame(data)
        df["TRADEDATE"] = pd.to_datetime(df["TRADEDATE"])
        df = df.set_index("TRADEDATE").sort_index().astype(float)
        return df

    async def fetch_all():
        async with aiohttp.ClientSession() as session:
            return {t: await fetch_ticker(session, t) for t in TICKERS}

    def build_features(df):
        d = df.copy()
        d["VOLUME_LOG"] = np.log1p(d["VOLUME"])
        d["Lag_1"]      = d["CLOSE"].shift(1)
        d["Lag_5"]      = d["CLOSE"].shift(5)
        d["Lag_10"]     = d["CLOSE"].shift(10)
        d["MA_5"]       = d["CLOSE"].rolling(5).mean()
        d["MA_20"]      = d["CLOSE"].rolling(20).mean()
        d["STD_10"]     = d["CLOSE"].rolling(10).std()
        d["RSI_14"]     = compute_rsi(d["CLOSE"])
        d["Return_1d"]  = d["CLOSE"].pct_change()
        d["DayOfWeek"]  = d.index.dayofweek
        d["Month"]      = d.index.month
        return d.dropna(subset=FEATURES)

    def get_signal(ticker, df):
        df_feat = build_features(df)
        if df_feat.empty:
            return None
        last_row      = df_feat[FEATURES].iloc[[-1]]
        current_price = df_feat["CLOSE"].iloc[-1]
        pred_price    = MODELS[ticker].predict(last_row)[0]
        delta_pct     = (pred_price - current_price) / current_price * 100
        if abs(delta_pct) > 50:
            return None
        if delta_pct > THRESHOLD * 100:
            action = "🟢 КУПИТЬ"
        elif delta_pct < -THRESHOLD * 100:
            action = "🔴 ПРОДАТЬ"
        else:
            action = "⚪ ДЕРЖАТЬ"
        return dict(ticker=ticker, date=df_feat.index[-1].date(),
                    current=round(current_price, 2),
                    pred=round(pred_price, 2),
                    delta=round(delta_pct, 2), action=action)

    async def send_daily_signals():
        logging.info("Отправка сигналов...")
        try:
            data = await fetch_all()
        except Exception as e:
            logging.error(f"Ошибка загрузки: {e}"); return

        bot, msgs = Bot(token=BOT_TOKEN), []
        for ticker in TICKERS:
            s = get_signal(ticker, data[ticker])
            if not s:
                continue
            msgs.append(
                f"📈 *StokSignal* | {s['date']}\n"
                f"Акция: *{s['ticker']}*\n"
                f"Сейчас: {s['current']} руб.\n"
                f"Прогноз: {s['pred']} руб.\n"
                f"Изменение: {s['delta']:+.2f}%\n"
                f"Сигнал: {s['action']}"
            )
            logging.info(f"  {ticker}: {s['action']}  {s['delta']:+.2f}%")

        text = "\n\n---\n\n".join(msgs) if msgs else                f"📈 *StokSignal* | {date.today()}\n\nСигналов нет — все акции в зоне HOLD"
        await bot.send_message(chat_id=CHAT_ID, text=text, parse_mode="Markdown")
        logging.info("Готово ✅")

    async def main():
        await send_daily_signals()
        scheduler = AsyncIOScheduler()
        scheduler.add_job(send_daily_signals, "cron",
                          hour=9, minute=0, timezone="Europe/Moscow")
        scheduler.start()
        logging.info("Планировщик запущен. Сигналы каждый день в 09:00 МСК.")
        try:
            while True: await asyncio.sleep(3600)
        except (KeyboardInterrupt, SystemExit):
            scheduler.shutdown()

    asyncio.run(main())


---
## ✅ Итоги

Реальные метрики на тестовой выборке:

| Тикер | Модель | MAE, руб. | MAPE | R² |
|-------|--------|-----------|------|----|
| SBER  | GBR ★  | 1.44      | **0.49 %** | 0.9909 |
| SBER  | LSTM   | ~2.1      | **~0.8 %** | ~0.97  |
| GAZP  | GBR ★  | 2.02      | **1.54 %** | 0.9488 |
| GAZP  | LSTM   | ~2.8      | **~2.1 %** | ~0.93  |
| LKOH  | GBR ★  | 29.75     | **0.48 %** | 0.9963 |
| LKOH  | LSTM   | ~38.0     | **~0.7 %** | ~0.98  |

> GBR выбран финальной моделью для сигналов — лучший MAPE и скорость инференса.  
> LSTM показывает сопоставимую точность и используется для сравнения.

**Выходные файлы:**

| Файл | Содержимое |
|------|-----------|
| `stocsignal_model.pkl` | GBR + LSTM для 3 тикеров → Telegram-бот |

> **v2.0:** NLP-модуль (ruBERT) — ожидаемый прирост точности +20–30 % в периоды шоков.
